To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

This notebook serves as a practical template for fine-tuning the Gemma 4 model for Audio Speech Recognition (ASR) using Unsloth. It covers data preparation, model training, inference, and saving the fine-tuned model. Students can adapt this workflow for various multimodal tasks by changing the dataset, model, and specific fine-tuning parameters.

### Installation

### Installation and Environment Setup

This section handles the setup of the necessary software libraries and environment configuration. It's crucial for ensuring all dependencies are met before running the model and training.

In [ ]:
# Installation & Environment Setup
# This cell installs all required dependencies for fine-tuning Gemma 4 with audio support.

%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

This code cell installs all the required Python packages for running this notebook, including Unsloth, HuggingFace libraries, and audio processing tools.

- `%%capture`: This magic command suppresses the output of the cell, making the notebook cleaner, especially during lengthy installations.
- The `if "COLAB_" not in "".join(os.environ.keys()):` checks if the notebook is running in a Google Colab environment. This allows for different installation commands based on the environment.
- `!pip install unsloth` (for local setups) vs `!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth` (for Colab): Unsloth provides optimized installation for Colab that leverages pre-compiled dependencies, leading to faster setup and better performance.
- `sentencepiece`, `protobuf`, `datasets`, `huggingface_hub`, `hf_transfer`: Core libraries for data handling and interacting with Hugging Face models.
- `torchao`: A library for PyTorch optimizations.
- `transformers` and `tokenizers`: Essential for working with large language models, providing model architectures and tokenization utilities.
- `torchcodec`: Specifically for audio codec functionalities, essential for Gemma 4's audio capabilities.
- `torch._dynamo.config.recompile_limit = 64`: This setting helps optimize PyTorch's `torch.compile` functionality, which Unsloth leverages for performance gains.

In [ ]:
# Install Extra Dependencies for Gemma 4 Vision & Audio
# Installs 'timm' (PyTorch Image Models), which is required by Gemma 4's multimodal architecture for processing audio and vision inputs.

%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

This cell installs `timm` (PyTorch Image Models), which is a library containing a collection of state-of-the-art image models. It's specifically required by Gemma 4's multimodal architecture for processing audio and vision inputs, even though the primary task here is ASR. This highlights Gemma 4's versatility in handling various data types.

- **Alternative:** For purely text-based tasks, `timm` might not be necessary, saving installation time and resources. However, for multimodal models like Gemma 4, it's often a prerequisite for full functionality.

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

### Unsloth Model Loading

This section focuses on loading the Gemma 4 model using Unsloth's `FastModel` class, which offers significant advantages in terms of memory efficiency and speed for fine-tuning.

In [ ]:
# Load the Gemma 4 Model
# Loads the **Gemma 4 E2B Instruct** model using Unsloth's 'FastModel',
# which provides optimized memory usage and faster training compared to loading via standard HuggingFace `transformers`.

from unsloth import FastModel
import torch
from huggingface_hub import snapshot_download

fourbit_models = [
    # Gemma 4 models
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B-it",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, processor = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

This cell loads the pre-trained Gemma 4 E2B Instruct model using `unsloth.FastModel`.

- **`FastModel.from_pretrained(...)`**: This is Unsloth's optimized method for loading models. It's designed to reduce memory usage and accelerate training compared to standard HuggingFace `transformers` loading.
- **`model_name = "unsloth/gemma-4-E2B-it"`**: Specifies the particular Gemma 4 model to load. Students can choose other Gemma 4 variants or even different models from the Unsloth Hugging Face page.
- **`dtype = None`**: Automatically detects the appropriate data type (e.g., `bfloat16`) for optimal performance on the available hardware.
- **`max_seq_length = 8192`**: Sets the maximum sequence length the model can process. A larger value allows for longer inputs (text or audio sequences), which is crucial for tasks like long-form audio transcription.
- **`load_in_4bit = False`**: When set to `True`, it loads the model weights in 4-bit quantization, drastically reducing GPU memory consumption. This is often crucial for fine-tuning large models on limited hardware. Here it's `False`, implying a larger precision (e.g., 16-bit) is used, which might be beneficial for accuracy, if memory permits. The output `Switching to 16bit LoRA` indicates that since `full_finetuning` is also `False`, it defaults to 16-bit LoRA.
- **`full_finetuning = False`**: When `True`, it enables full fine-tuning of all model parameters. When `False`, it typically enables Parameter-Efficient Fine-Tuning (PEFT) methods like LoRA, which only tune a small subset of parameters, saving memory and speeding up training.
- **`token = "YOUR_HF_TOKEN"`**: (Commented out) If the chosen model is gated on Hugging Face, a Hugging Face token is required for access.

# Gemma 4 can process Text, Vision and Audio!

Let's first experience how Gemma 4 can handle multimodal inputs. We use Gemma 4's recommended settings of `temperature = 1.0, top_p = 0.95, top_k = 64` but for this example we use `do_sample=False` for ASR.

### Multimodal Inference Setup

This section prepares a helper function for performing inference with the Gemma 4 model, demonstrating its capability to handle multimodal inputs (in this case, audio) and generate responses. It also highlights recommended inference settings.

In [ ]:
# Define the Inference Helper Function
# Defines 'do_gemma_4_inference()' — a reusable helper that:
# 1. Applies the Gemma 4 chat template to a list of 'messages' (formats system/user/assistant turns into the correct token structure)
# 2. Moves inputs to GPU ('cuda')
# 3. Runs greedy decoding (`do_sample=False`) for deterministic ASR output
# 4. Streams output tokens to the screen in real-time using `TextStreamer`


from transformers import TextStreamer
# Helper function for inference
def do_gemma_4_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **processor.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        do_sample = False,
        streamer = TextStreamer(processor, skip_prompt = True),
    )

This cell defines a crucial helper function, `do_gemma_4_inference`, to streamline the process of running inferences with the Gemma 4 model. This function encapsulates the steps needed to prepare input messages and generate output.

- **`messages`**: This parameter takes a list of dictionaries representing the conversational turns. Each dictionary specifies a `role` (e.g., "system", "user", "assistant") and `content`. For multimodal inputs, `content` is a list of dictionaries that can include both `"type": "audio"` (with the audio array) and `"type": "text"`.
- **`processor.apply_chat_template(...)`**: This method is vital for formatting the `messages` list into the specific token structure that the Gemma 4 model expects. This includes special tokens for roles and multimodal inputs.
    - `add_generation_prompt = True`: This is essential for generation tasks, as it adds the necessary prompt to encourage the model to generate a response.
    - `tokenize = True`: Converts the text and audio inputs into numerical tokens the model understands.
    - `return_dict = True` and `return_tensors = "pt"`: Specifies that the output should be a dictionary of PyTorch tensors, suitable for model input.
    - `.to("cuda")`: Moves the processed inputs to the GPU for faster computation.
- **`max_new_tokens = 128`**: Limits the number of new tokens the model will generate, controlling the length of the response.
- **`do_sample = False`**: When `False`, the model uses greedy decoding, always selecting the token with the highest probability. This results in deterministic output, which is often preferred for tasks like ASR where a single, most probable transcription is desired. For creative generation, `do_sample = True` would introduce randomness.
- **`streamer = TextStreamer(processor, skip_prompt = True)`**: This enables streaming output, where tokens are printed to the console as they are generated, giving real-time feedback. `skip_prompt = True` prevents the input prompt from being re-printed in the output.

<h3>Let's Evaluate Gemma 4 Baseline Performance on German Transcription</h2>

In [ ]:
# Load the Training Dataset
# Loads the **Emilia German speech dataset** ('kadirnar/Emilia-DE-B000000') from HuggingFace Hub — a large multilingual speech corpus. We use the German subset for this ASR fine-tuning task.

# **What this cell does:**
# 1. Loads the full training split
# 2. **Reserves sample index '7546'** as a held-out test example — this is done *before* slicing so it's never seen during training
# 3. **Slices the dataset to the first 3,000 samples** to keep training fast and VRAM-friendly
# 4. **Resamples audio to 16kHz** — required by Gemma 4's audio encoder

from datasets import load_dataset,Audio,concatenate_datasets

dataset = load_dataset("kadirnar/Emilia-DE-B000000", split = "train")

# Select a single audio sample to reserve for testing.
# This index is chosen from the full dataset before we create the smaller training split.
test_audio = dataset[7546]

dataset = dataset.select(range(3000))

dataset = dataset.cast_column("audio", Audio(sampling_rate = 16000))

README.md:   0%|          | 0.00/540 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/495M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/503M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12038 [00:00<?, ? examples/s]

In [ ]:
# Step 6: Preview the Held-Out Test Sample

# Plays back the reserved test audio sample and prints its ground-truth transcript so you can:
# - **Verify the dataset loaded correctly** (audio is audible and text matches)
# - **Have a reference** to compare against the model's transcription output before and after fine-tuning

from IPython.display import Audio, display
print(test_audio['text'])
Audio(test_audio['audio']['array'],rate = test_audio['audio']['sampling_rate'])

 Ich, ich rechne direkt mich an. Das ist natürlich klar, nur, dass, äh, es politische Interessen gibt im Handel, im Austausch mit Waren, dass es politische Einflüsse gibt. Die Frage ist, die Alternative soll es nicht sein.


And the translation of the audio from German to English is:

> I—I hold myself directly accountable. That much is, of course, clear: namely, that there are political interests involved in trade—in the exchange of goods—and that political influences are at play. The question is: that should not be the alternative.

In [ ]:
# Preview the Held-Out Test Sample

messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an assistant that transcribes speech accurately.",
            }
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": test_audio['audio']['array']},
            {"type": "text", "text": "Please transcribe this audio."}
        ]
    }
]

do_gemma_4_inference(messages, max_new_tokens = 256)

Sie sich direkt mich an und ist nicht klar, was politische Interessen gibt im Handel, im Austausch mit Waren, dass es politische Einflüsse gibt. Die globale ist die Alternative soll es nicht sein.<turn|>


<h3>Baseline Model Performance: 32.43% Word Error Rate (WER) for this sample !</h3>

# Let's finetune Gemma 4!

You can finetune the vision and text and audio parts

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
# Apply LoRA Adapters for Parameter-Efficient Fine-Tuning
# Wraps the model with **LoRA (Low-Rank Adaptation)** so only a small fraction of parameters are updated during training,
# dramatically reducing VRAM and training time.

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True,  # False if not finetuning language layers
    finetune_attention_modules = True,  # False if not finetuning attention layers
    finetune_mlp_modules       = True,  # False if not finetuning MLP layers

    r = 8,                              # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,                    # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,                 # We support rank stabilized LoRA
    loftq_config = None,                # And LoftQ
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",

        # Audio layers
        "post", "linear_start", "linear_end",
        "embedding_projection",
        "ffw_layer_1", "ffw_layer_2",
        "output_proj",
    ]
)

<a name="Data"></a>
### Data Prep
We adapt the `kadirnar/Emilia-DE-B000000` dataset for our German ASR task using Gemma 4 multi-modal chat format. Each audio-text pair is structured into a conversation with `system`, `user`, and `assistant` roles. The processor then converts this into the final training format:

```
<bos><|turn>system
You are an assistant that transcribes speech accurately.<turn|>
<|turn>user
<|audio|>Please transcribe this audio.<turn|>
<|turn>model
Ich, ich rechne direkt mich an.<turn|>

In [ ]:
# Format Dataset into Chat Template
# Data Preparation
# Converts the raw '(audio, text)' pairs from the dataset into the **Gemma 4 multimodal chat format** required for supervised fine-tuning.


def format_intersection_data(samples: dict) -> dict[str, list]:
    """Format intersection dataset to match expected message format"""
    formatted_samples = {"messages": []}
    for idx in range(len(samples["audio"])):
        audio = samples["audio"][idx]["array"]
        label = str(samples["text"][idx])

        message = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": "You are an assistant that transcribes speech accurately.",
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio": audio},
                    {"type": "text", "text": "Please transcribe this audio."}
                ]
            },
            {
                "role": "assistant",
                "content":[{"type": "text", "text": label}]
            }
        ]
        formatted_samples["messages"].append(message)
    return formatted_samples

In [ ]:
#  Apply Formatting to the Full Dataset
# Applies the 'format_intersection_data()' function across all 3,000 training samples using batched mapping.

# **Parameters:**
# - 'batched=True' — processes samples in batches for speed
# - 'batch_size=4' — small batch size keeps audio arrays manageable in memory
# - 'num_proc=4' — uses 4 CPU workers in parallel to speed up formatting

dataset = dataset.map(format_intersection_data, batched = True, batch_size = 4, num_proc = 4)

Map (num_proc=4):   0%|          | 0/3000 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
# Use UnslothVisionDataCollator which handles audio token alignment correctly
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 1,
        warmup_ratio = 0.03,
        # num_train_epochs = 1, # Use for full training runs
        max_steps = 60,
        learning_rate = 5e-5,
        logging_steps = 1,
        save_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,

        # The below are a must for audio finetuning:
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 8192,
    )
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


In [ ]:
# Check GPU Memory Before Training

# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
9.646 GB of memory reserved.


# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
# Start Training
# Launches the fine-tuning run using the configured trainer.

trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 18,237,440 of 5,141,415,456 (0.35% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
1,15.779630
2,16.243879
3,15.789350
4,15.544473
5,16.026503
6,16.052254
7,15.395693
8,14.153306
9,13.713164
10,11.837756


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [ ]:
# Review Memory & Time Stats After Training

# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

152.4949 seconds used for training.
2.54 minutes used for training.
Peak reserved memory = 11.145 GB.
Peak reserved memory for training = 1.499 GB.
Peak reserved memory % of max memory = 76.53 %.
Peak reserved memory for training % of max memory = 10.293 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-4` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64` but for this example we use `do_sample=False` for ASR.

In [ ]:
# Post-Training Inference
# Runs the **fine-tuned model** on the held-out test sample to evaluate improvement.

messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an assistant that transcribes speech accurately.",
            }
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": test_audio['audio']['array']},
            {"type": "text", "text": "Please transcribe this audio."}
        ]
    }
]

do_gemma_4_inference(messages, max_new_tokens = 256)

Sie sich direkt mich an und ist nicht klar, was politische Interessen gibt im Handel, im Austausch mit Waren, dass es politische Einflüsse gibt. Die globale ist die Alternative soll es nicht sein.<turn|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
# Save the Fine-Tuned LoRA Adapters

model.save_pretrained("gemma_4_lora")  # Local saving
processor.save_pretrained("gemma_4_lora")
# model.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving
# processor.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving

Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/tokenizer_config.json.


['gemma_4_lora/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, processor = FastModel.from_pretrained(
        model_name = "gemma_4_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(processor, skip_prompt = True),
)

I am Gemma 4, a Large Language Model developed by Google DeepMind. I am an open weights model.<turn|>


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-4-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-4", processor)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-finetune", processor,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma_4_finetune",
        processor,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma_4_finetune",
        processor,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )

Now, use the `gemma-4-finetune.gguf` file or `gemma-4-finetune-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).